In [ ]:
import glob
import os
from pathlib import Path

import numpy as np
from cogent3 import open_data_store
from plot_utils.project_paths import DATA_DIR, FIG_DIR, write_pdf
from plot_utils.util import load_json_app

DATA_DIR = Path(DATA_DIR)

## 1. Theoretical relationship between non-statioanrity difference & clock violation



In [ ]:
from plot_utils.theoretical_analysis import (
    plot_time_grouped_scatter_2x2,
    compute_aginst_Q,
    correlation_factor_plot)

In [ ]:
import json
valid_matrix_full_path =  DATA_DIR / 'valid_matrix_full.json'
valid_matrix_full = json.load(open(valid_matrix_full_path, 'r'))
matrices_list = []
for gene, matrices in valid_matrix_full.items():
    matrices_list.extend(matrices)

In [ ]:
len(matrices_list)

In [ ]:
t_range = [0.5, 1, 1.5, 2]
pi0 = [0.25, 0.25, 0.25, 0.25]
#0.45
pi4 = [0.06, 0.47, 0.08, 0.39]
df_low = compute_aginst_Q(matrices_list, pi0, t_range)
df_high = compute_aginst_Q(matrices_list, pi4, t_range)

In [ ]:
time_jsd_diff_fig_low = plot_time_grouped_scatter_2x2(df_low, 'JSD_difference', 'ENS_abs_difference')
# write_pdf(time_jsd_diff_fig_low, FIG_DIR / "Theoretical_relationship_low_entropy.pdf")

time_jsd_diff_fig_high = plot_time_grouped_scatter_2x2(df_high, 'JSD_difference', 'ENS_abs_difference')
# write_pdf(time_jsd_diff_fig_high, FIG_DIR / "Theoretical_relationship_high_entropy.pdf")


In [ ]:
correlation_factor_fig = correlation_factor_plot(df_low, df_high, t_range)
write_pdf(correlation_factor_fig, FIG_DIR / "Theoretical_relationship_correlation_factor.pdf")
correlation_factor_fig.show()

## 2. Test of clock violation and non-stationarty evolution in mammal genome


In [ ]:
from plot_utils.bootstrapping import (
    get_p_value_distribution_tos,
    get_p_value_distribution_toc,
    qq_plot_null_observed,
    qq_plot_uniform,
    get_rejected_proportion,
    get_proportion_rejected_correlation_fig
)

 

In [ ]:
from mdeq.utils import estimate_freq_null

### Test of stationarity

In [ ]:
tos_result_dir =  DATA_DIR / 'tos_boostrapping_results'
input_data_store_tos_result = open_data_store(tos_result_dir, suffix= 'json')
p_value_observed_dict_tos, p_value_tested_dict_tos, p_value_list_tos_null_tos = get_p_value_distribution_tos(input_data_store_tos_result)
p_value_observed_dict_tos_updated = [a for a in p_value_observed_dict_tos.values() if a is not None]
smile_fig_tos = qq_plot_null_observed(list(p_value_list_tos_null_tos.values()), p_value_observed_dict_tos_updated)


In [ ]:
smile_fig_tos.show()

In [ ]:
estimate_freq_null_tos = estimate_freq_null(np.array(list(p_value_tested_dict_tos.values())))
1-estimate_freq_null_tos

In [ ]:
# write_pdf(smile_fig_tos, FIG_DIR / "Bootstrapping_smile_fig_tos.pdf", width= 400, height= 400)

In [ ]:
example_p_value_tos_uniform_null = []
example_tos_bootstrape_result = load_json_app(input_data_store_tos_result[100])
for key in example_tos_bootstrape_result.keys():
    if key != 'observed':
        example_p_value_tos_uniform_null.append(example_tos_bootstrape_result[key].pvalue)
qqplot_tos = qq_plot_uniform([p for p in example_p_value_tos_uniform_null if p != None])


In [ ]:
# write_pdf(qqplot_tos, FIG_DIR / "Bootstrapping_qqplot_tos.pdf", width= 400, height= 400)


### Test of molecular clock

In [ ]:
#NOTE: this takes pretty long time to run so I have saved the results into json file 
# input_dstore = open_data_store(DATA_DIR / "triples_representative_subset_json", suffix="json")
# p_value_list_null = []
# for data in input_dstore:
#     gene_name = data.unique_id.split('.')[0]
#     aln = load_json_app(data)
#     result = hyp_test_clock(aln)
#     null_result = result.null
#     simulate_aln = null_result.lf.simulate_alignment()
#     simulate_aln.info["triples_species_name"] = aln.info["triples_species_name"]
#     simulate_hyp_result = hyp_test_clock(simulate_aln)
#     p_value_list_null.append(simulate_hyp_result.pvalue)

# with open ('/Users/gulugulu/clock/mammal_orthologs_hsap_1/toc_hypothesis_test_results_pvalue.json', 'w') as outfile:
#     json.dump(p_value_list_null, outfile, indent = 4)

In [ ]:
toc_result_dir =  DATA_DIR / 'toc_hypothesis_test_results'
p_value_list_toc_null_toc_dir = DATA_DIR / 'toc_hypothesis_test_results_pvalue.json'

with open(p_value_list_toc_null_toc_dir, 'r', encoding='utf-8') as infile:
    p_value_list_toc_null_toc = json.load(infile)
p_value_list_toc_null_toc_updated = [a for a in p_value_list_toc_null_toc if a is not None]
input_data_store_toc_result = open_data_store(toc_result_dir, suffix= 'json')
p_value_observed_dict_toc = get_p_value_distribution_toc(input_data_store_toc_result)
p_value_observed_dict_toc_updated = [a for a in p_value_observed_dict_toc.values() if a is not None]
smile_fig_toc = qq_plot_null_observed(p_value_list_toc_null_toc_updated, p_value_observed_dict_toc_updated)


In [ ]:
# write_pdf(smile_fig_toc, FIG_DIR / "Bootstrapping_smile_toc.pdf", width= 400, height= 500)
# write_pdf(smile_fig_tos, FIG_DIR / "Bootstrapping_smile_tos.pdf", width= 400, height= 500)


In [ ]:
len([p for p in p_value_observed_dict_toc_updated if p < 0.05])/len(p_value_observed_dict_toc_updated)

In [ ]:
estimate_freq_null_toc = estimate_freq_null(np.array(p_value_observed_dict_toc_updated))
1-estimate_freq_null_toc

In [ ]:
# write_pdf(smile_fig_toc, FIG_DIR / "Bootstrapping_smile_fig_toc.pdf", width= 400, height= 400)


In [ ]:
# example_p_value_toc_uniform_null = []
# example_toc_bootstrape_result = load_json_app(input_data_store_toc_result[100])
# for key in example_toc_bootstrape_result.keys():
#     if key != 'observed':
#         example_p_value_toc_uniform_null.append(example_toc_bootstrape_result[key].pvalue)
# qqplot_toc = qq_plot_uniform([p for p in example_p_value_toc_uniform_null if p != None])
# write_pdf(qqplot_toc, FIG_DIR / "Bootstrapping_qqplot_toc.pdf", width= 400, height= 600)


In [ ]:
proportion_rejected_tos = get_rejected_proportion(p_value_tested_dict_tos)
proportion_rejected_toc = get_rejected_proportion(p_value_observed_dict_toc)

In [ ]:
rejected_proportion_correlation_plot = get_proportion_rejected_correlation_fig(proportion_rejected_tos, proportion_rejected_toc)

In [ ]:
rejected_proportion_correlation_plot.show()
write_pdf(rejected_proportion_correlation_plot, FIG_DIR / "Bootstrapping_proportion_rejected_correlation.pdf", width= 600, height= 400)


In [ ]:
from scipy.stats import spearmanr
proportion_rejected_toc_filtered = {gene: proportion_rejected_toc[gene] for gene in proportion_rejected_tos.keys()}
spearmanr(list(proportion_rejected_tos.values()), list(proportion_rejected_toc_filtered.values()))

In [ ]:
# write_pdf(rejected_proportion_correlation_plot, FIG_DIR / "Bootstrapping_proprotion_rejected_correlation.pdf", width= 800, height= 600)


## 3. Empirical relationship between non-statioanrity difference & clock violation in mammal genome

In [ ]:
from plot_utils.empirical_analysis import (
    get_gene_value_dict,
    spearman_correlation_analysis,
    get_correlation_factors_distplot,
    get_correlation_factor_tos_rejection_correlation_fig
)



In [ ]:
triples_model_fitting_dir =  DATA_DIR / 'triples_model_fitting'
gene_paths = glob.glob(os.path.join(triples_model_fitting_dir, '*/'))


gene_value_dict = get_gene_value_dict(gene_paths)
p_value_list, p_value_list_corrected, correlation_list = spearman_correlation_analysis(gene_value_dict)

correlation_factor_dist_plot = get_correlation_factors_distplot(correlation_list)
# write_pdf(correlation_factor_dist_plot, FIG_DIR / "Empirical_relationship_correlation_strength_distirbution.pdf", width= 800, height= 600)

correlation_factor_tos_rejection_correlation_fig = get_correlation_factor_tos_rejection_correlation_fig(proportion_rejected_tos, correlation_list)
# write_pdf(correlation_factor_tos_rejection_correlation_fig, FIG_DIR / "Empirical_relationship_correlation_strength_tos_reject_proportion_correlation.pdf",width= 800, height= 600)


In [ ]:
correlation_factor_dist_plot.show()
write_pdf(correlation_factor_dist_plot, FIG_DIR / "Empirical_relationship_correlation_strength_distirbution.pdf", width= 600, height= 500)


In [ ]:


correlation_factor_tos_rejection_correlation_fig.show()
write_pdf(correlation_factor_tos_rejection_correlation_fig, FIG_DIR / "Empirical_relationship_correlation_strength_tos_reject_proportion_correlation.pdf",width= 600, height= 400)

In [ ]:
len([a for a in list(correlation_list.values()) if a > 0.4])/len(correlation_list)

In [ ]:
from scipy.stats import spearmanr
proportion_rejected_tos_filtered = {gene: proportion_rejected_tos[gene] for gene in correlation_list.keys()}
spearmanr(list(correlation_list.values()), list(proportion_rejected_tos_filtered.values()))

## 4. Mathematical association between non-stationary evolution and clock violation

In [ ]:
from plot_utils.math import (
    get_evolutionary_rate_change_plot,
    get_ens_plot,
    get_evolutionary_rate_plot
)

In [ ]:
t_range = np.linspace(0, 10, 999)
Q1 = np.array([[-0.14, 0.01, 0.04, 0.09], 
           [0.4, -0.69, 0.09, 0.2], 
           [0.63, 0.2, -1.3, 0.3], 
           [0.07,0.01, 0.02, -0.1]])*0.3

Q2 = np.array([[-0.242,  0.197,  0.008,  0.036],
        [ 0.17 , -0.204,  0.004,  0.03 ],
        [ 0.044,  0.067, -0.137,  0.025],
        [ 0.018,  0.017,  0.146, -0.18 ]])

Q3= np.array([[-0.633,  0.528,  0.006,  0.099],
        [ 0.111, -0.174,  0.017,  0.046],
        [ 0.133,  0.094, -0.312,  0.085],
        [ 0.028,  0.023,  0.045, -0.096]])
pi = np.array([0.05, 0.35, 0.35, 0.25])


In [ ]:
## Stationary process
stationary_ENS_plot = get_ens_plot(Q1, t_range)
# write_pdf(stationary_ENS_plot, FIG_DIR / "Stationary_ENS_plot.pdf", width = 600, height = 400)

stationary_evolutionary_rate_plot = get_evolutionary_rate_plot(Q1, t_range)
# write_pdf(stationary_evolutionary_rate_plot, FIG_DIR / "Stationary_evolutionary_rate_plot.pdf", width = 600, height = 400)


In [ ]:
stationary_ENS_plot.show()

In [ ]:
stationary_evolutionary_rate_plot.show()

In [ ]:
## Non stationary process
non_stationary_ENS_plot = get_ens_plot(Q1, t_range, pi)
# write_pdf(non_stationary_ENS_plot, FIG_DIR / "Non_stationary_ENS_plot.pdf", width = 600, height = 400)

non_stationary_evolutionary_rate_plot = get_evolutionary_rate_plot(Q1, t_range, pi)
# write_pdf(non_stationary_evolutionary_rate_plot, FIG_DIR / "Non_stationary_evolutionary_rate_plot.pdf", width = 600, height = 400)



In [ ]:
non_stationary_ENS_plot.show()

In [ ]:
non_stationary_evolutionary_rate_plot.show()

In [ ]:
non_stationary_evolutionary_rate_change_plot = get_evolutionary_rate_change_plot(Q1,Q2,Q3, pi, t_range)
write_pdf(non_stationary_evolutionary_rate_change_plot, FIG_DIR / "Non_stationary_evolutionary_rate_change_plot.pdf", width = 1100, height = 500)
non_stationary_evolutionary_rate_change_plot.show()